In [1]:
import pandas as pd
from train import make_train_fn
from experiment import greedy_symbolic_search
from typing import List, Dict, Tuple, Optional
import os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_distances


In [2]:
df = pd.read_csv("../data/ait_ads/combined_ait_labeled.csv", low_memory=False)
df.columns

Index(['timestamp', 'category', 'entity', 'raw_log', 'scenario', 'source',
       'host_ip', 'host', 'rule_id', 'rule_desc', 'groups', 'groups_raw',
       'groups_str', 'srcip', 'dstip', 'srcport', 'dstport', 'proto',
       'is_auth_event', 'is_cred_event', 'is_web_event', 'is_cron',
       'is_success', 'is_uid0', 'aminer_component_type',
       'aminer_training_mode', 'aminer_new_event', 'wazuh_level',
       'wazuh_antivirus', 'wazuh_update', 'is_ids_alert', 'ids_signature',
       'ids_category', 'ids_severity', 'exploit_class', 'alert_channel',
       'decoder', 'decoder_parent', 'location', 'mitre_ids', 'mitre_tactic',
       'mitre_technique', 'username', 'procname', 'event_label', 'y'],
      dtype='object')

In [3]:
# ### Greedy search on "best" symbolic feature subsets?
# window_size = pd.Timedelta(days=1)
# train_fn = make_train_fn(test_frac=0.3)

# search_res = greedy_symbolic_search(
#     df, window_size, train_fn=train_fn, threshold=0.5, max_k=6
# )

# print("Baseline metrics:", search_res["base_metrics"])
# print("Chosen symbolic subset:", search_res["chosen"])
# print("History:")
# for row in search_res["history"]:
#     print(row)

### Normalize `category`

In [4]:
df["category_norm"] = df["category"].astype(str).str.split(" - ").str[0].str.strip()

## Tokenize alerts

In [5]:
def tokenize_alerts(
    df: pd.DataFrame,
    fields: List[str],
    source_col: str = "source",          # e.g. "aminer" / "wazuh"
    add_source_prefix: bool = True,
    null_token: str = "<NA>",
    max_unique_per_field: int = 5000,    # guardrail against explosions (IPs, paths, etc.)
) -> pd.Series:
    """
    Returns a Series of list-of-tokens per row.
    Each token is like: "wazuh:rule_level=7" or "aminer:component=ssh".
    """
    # Guardrail: drop fields that don't exist
    fields = [f for f in fields if f in df.columns]

    # Optional: cap high-cardinality fields to avoid blowing up tokens
    usable_fields = []
    for f in fields:
        nunique = df[f].nunique(dropna=True)
        if nunique <= max_unique_per_field:
            usable_fields.append(f)

    src = df[source_col].fillna("unknown") if source_col in df.columns else pd.Series(["unknown"] * len(df), index=df.index)

    tokens_per_row = []
    for idx, row in df.iterrows():
        row_tokens = []
        src_prefix = f"{row[source_col]}:" if (add_source_prefix and source_col in df.columns) else ""
        for f in usable_fields:
            val = row[f]

                # skip NA and emptry string for some base fields
            if pd.isna(val) or str(val).strip() == "":
                    continue
            if pd.isna(val):
                v = null_token
            else:
                v = str(val).strip()
                if not v:
                    v = null_token
            row_tokens.append(f"{src_prefix}{f}={v}")
        tokens_per_row.append(row_tokens)

    return pd.Series(tokens_per_row, index=df.index)


In [6]:
base_fields = [
    # shared / general
    "source", "scenario",
    "category_norm", "decoder", "decoder_parent",
    "host", "host_ip",
    "proto",

    # wazuh
    "wazuh_level",
    "groups_str",          
    "is_ids_alert",
    "ids_category",
    "ids_severity",
    "mitre_tactic",
    "mitre_technique",

    # aminer
    "aminer_component_type",
    "aminer_training_mode",
    "aminer_new_event",

    # auth/web context flags (already abstracted!)
    "is_auth_event", "is_web_event", "is_cred_event",
    "is_success", "is_uid0",
]

min_support=20


## Mine tokens

In [7]:
def mine_fp_contrast_tokens(
    tokens: pd.Series,          # list-of-tokens per row
    y: pd.Series,               # 0 benign, 1 attack
    top_k: int = 50,
    min_support: int = 200,     # only keep tokens that appear at least this many times overall
    alpha: float = 0.5,         # smoothing for log-odds
) -> pd.DataFrame:
    """
    Returns a dataframe of tokens ranked by how benign-associated they are (label 0).
    Score: smoothed log-odds(token | benign) - log-odds(token | attack)
          = log( (c0+α)/(n0-c0+α) ) - log( (c1+α)/(n1-c1+α) )
    Higher => more benign/FP-associated.
    """
    # Flatten tokens: one row per token ocurrence
    flat = tokens.explode()
    flat_y = y.loc[flat.index]

    # count token occurences for benign vs attack
    n0 = int((flat_y == 0).sum())
    n1 = int((flat_y == 1).sum())

    # Count token frequency per class
    c0 = flat[flat_y == 0].value_counts()
    c1 = flat[flat_y == 1].value_counts()

    all_tokens = c0.index.union(c1.index)
    c0 = c0.reindex(all_tokens, fill_value=0)
    c1 = c1.reindex(all_tokens, fill_value=0)

    # filter out rarely appearing tokens
    total = c0 + c1
    keep = total[total >= min_support].index
    c0 = c0.loc[keep]
    c1 = c1.loc[keep]
    total = total.loc[keep]

    # Smoothed log-odds contrast
    # how likely is the token in class benign vs how likely is it in attack
    # positive = token shows up mostly in benign = FP suppression candidate
    # near zero = token appears similarly
    # negative = token is attack-associated
    score = np.log((c0 + alpha) / ((n0 - c0) + alpha)) - np.log((c1 + alpha) / ((n1 - c1) + alpha))

    out = pd.DataFrame({
        "token": c0.index,
        "count_benign": c0.values,
        "count_attack": c1.values,
        "support_total": total.values,
        "score_fp_contrast": score.values,
        "p_benign_given_token": (c0 / (c0 + c1)).values,
    }).sort_values("score_fp_contrast", ascending=False)

    # keep only top k tokens
    # return out.head(top_k).reset_index(drop=True)
    return out


In [8]:
def plot_token_semantic_scatter(
    ranking: pd.DataFrame,
    min_support,
    token_col: str = "token",
    p_col: str = "p_benign_given_token",
    benign_thresh: float = 0.60,
    attack_thresh: float = 0.40,
    max_points: int = 5000,
    random_state: int = 0,
):
    """
    Semantic layout: TF-IDF (char ngrams) -> cosine distances -> t-SNE 2D
    Coloring: 3 categories based on p_benign_given_token
      - benign: p >= benign_thresh
      - attack: p <= attack_thresh
      - neutral: otherwise
    """

    df = ranking[[token_col, p_col]].dropna().copy()

    # Optional: cap points for speed (keeps most supported ones if present)
    if "support_total" in ranking.columns:
        df = ranking[[token_col, p_col, "support_total"]].dropna().sort_values(
            "support_total", ascending=False
        )
        df = df.head(max_points).copy()
    else:
        df = df.head(max_points).copy()

    tokens = df[token_col].astype(str).tolist()
    p = df[p_col].astype(float).to_numpy()

    # 1) "Semantic" representation (works well for short strings like tokens)
    vec = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=1)
    X = vec.fit_transform(tokens)

    # 2) Pairwise cosine distance (t-SNE can use precomputed distances)
    D = cosine_distances(X)

    # 3) 2D embedding
    perplexity = min(30, max(5, (len(tokens) - 1) // 3))
    tsne = TSNE(
        n_components=2,
        metric="precomputed",
        perplexity=perplexity,
        init="random",
        learning_rate="auto",
        random_state=random_state,
    )
    Z = tsne.fit_transform(D)

    # 4) 3 categories from p_benign_given_token
    labels = np.full(len(p), "neutral", dtype=object)
    labels[p >= benign_thresh] = "benign"
    labels[p <= attack_thresh] = "attack"

    # 5) Plot (no manual colors; matplotlib picks defaults)
    plt.figure()
    for lab in ["benign", "neutral", "attack"]:
        m = labels == lab
        plt.scatter(Z[m, 0], Z[m, 1], s=12, alpha=0.7, label=lab)

    plt.title(f"Semantic scatter of tokens (min_support={min_support})")
    plt.xlabel("dim 1")
    plt.ylabel("dim 2")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return df.assign(tsne_x=Z[:, 0], tsne_y=Z[:, 1], category=labels)

In [9]:
# df_plot = plot_token_semantic_scatter(ranking, min_support)

In [10]:
# df_plot.columns

In [11]:
def plot_class_histogram(df, label_col="y"):
    counts = df[label_col].value_counts().sort_index()

    plt.figure()
    plt.bar(counts.index.astype(str), counts.values)
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.title("Class Distribution")
    plt.tight_layout()
    plt.show()

# plot_class_histogram(df_plot, label_col="category")

## Add behavioural mining (window based)

In [12]:
def add_behavioral_features(df, time_col="timestamp", src_col="srcip", dst_col="dstip"):
    df = df.sort_values(time_col).copy()

    # Source frequency in window
    src_counts = df[src_col].value_counts()
    df["count_src_window"] = df[src_col].map(src_counts)

    # Bin it (important to avoid explosion)
    df["src_freq_bin"] = pd.cut(
        df["count_src_window"],
        bins=[-1, 5, 20, 100, float("inf")],
        labels=["low", "medium", "high", "very_high"]
    )

    # Destination fan-in
    fan_in = df.groupby(dst_col)[src_col].nunique()
    df["unique_src_per_dst"] = df[dst_col].map(fan_in)

    df["dst_fanin_bin"] = pd.cut(
        df["unique_src_per_dst"],
        bins=[-1, 3, 10, 50, float("inf")],
        labels=["low", "medium", "high", "very_high"]
    )

    return df


In [13]:
df_copy = add_behavioral_features(df)
tokens = tokenize_alerts(df_copy, base_fields + ["src_freq_bin", "dst_fanin_bin"])
ranking = mine_fp_contrast_tokens(tokens, df["y"])

In [14]:
os.makedirs("../out/ait_ads/tokens", exist_ok=True)
tokens.to_csv("../out/ait_ads/tokens/labeled_tokenized.csv")

In [15]:
ranking.shape

(240, 6)

In [16]:
benign_side = ranking.sort_values("score_fp_contrast", ascending=False).head(200)
attack_side = ranking.sort_values("score_fp_contrast", ascending=True).head(50)

In [17]:
attack_side.value_counts

<bound method DataFrame.value_counts of                                                  token  count_benign  \
192          wazuh:ids_category=Web Application Attack             0   
15   aminer:category_norm=AMiner: New request metho...           107   
35                      aminer:host_ip=192.168.104.155           127   
24                           aminer:host_ip=10.143.2.4           128   
193                             wazuh:ids_severity=1.0            24   
6    aminer:aminer_component_type=NewMatchPathValue...          4719   
38                      aminer:host_ip=192.168.188.179           186   
31                        aminer:host_ip=172.21.241.88           197   
55                     aminer:scenario=russellmitchell           880   
56                              aminer:scenario=santos           983   
39                        aminer:host_ip=192.168.2.114          4459   
52                               aminer:is_web_event=1         19027   
7                       

In [ ]:
benign_side.head()